In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="19gF2C722Y0Ev03IqVDsCKhZI1o141zLo", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/04_01_intro.mp3"))

# Batch API & Multi-Pass Review Architectures

**Notebook 4 of 4** | Prompt Engineering & Structured Output | Claude Certified Architect Prep

---

**Objective:** Structure batch requests for 50% cost savings on bulk workloads, and design multi-pass review architectures that separate detection from severity assessment from cross-file integration.

**Exam Tasks Covered:**
- **4.5** — Batch API: 50% cost, 24-hour window, custom_id tracking
- **4.6** — Multi-pass review: separate passes for detection, severity, cross-file integration

**Prerequisites:** Notebooks 01-03 (Criteria, tool_use, Validation-Retry)

---

In Notebooks 01-03, we built the core extraction toolkit: explicit criteria, few-shot examples, tool_use schemas, and validation-retry loops. Now we scale up. The **Batch API** processes thousands of documents at half the cost. **Multi-pass review** breaks complex analysis into specialized passes that each do one thing well.

By the end of this notebook, you will be able to:
1. Structure batch requests with `custom_id` for result tracking
2. Process batch results and match them back to source documents
3. Choose batch vs real-time based on latency and cost requirements
4. Design multi-pass review pipelines with specialized prompts per pass
5. Combine batch processing with multi-pass architecture

In [ ]:
#@title 🎧 Listen: Batch Overview
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_02_batch_overview.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Multipass Overview
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_03_multipass_overview.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Concept Overview

This notebook covers two techniques that scale prompt engineering from single-document processing to production workloads: the **Batch API** and **multi-pass review architectures**.

---

### Batch API

The Batch API lets you submit up to **100,000 requests** in a single job, with two major advantages:

| Feature | Real-Time API | Batch API |
|---------|--------------|-----------|
| **Cost** | Full price | **50% discount** |
| **Latency** | Seconds | Up to **24 hours** |
| **Concurrency** | Rate-limited | Managed by Anthropic |
| **Tracking** | Your own logic | Built-in `custom_id` |

**How it works:**
1. You build a list of requests, each with a unique `custom_id` and the same `params` you would send to `messages.create()`
2. Submit the batch — Anthropic queues and processes all requests within 24 hours
3. Poll for completion, then download results — each result carries its `custom_id` so you can match it back to the source document
4. Input/output uses **JSONL** format (one JSON object per line)

**When to use batch:**
- Overnight document processing (invoices, contracts, reports)
- Periodic bulk extraction (nightly data pipeline)
- Cost-sensitive workloads where next-day results are acceptable
- Large-scale evaluation or testing runs

**When to use real-time:**
- User-facing chat or interactive tools
- Live code review in an IDE
- Any workflow where the user is waiting for a response

---

### Multi-Pass Review Architecture

A single prompt asking "review this code" tries to do too many things at once: find bugs, assess severity, check style, consider cross-file context. The result is mediocre at everything.

**Multi-pass review** decomposes this into specialized passes, each with its own prompt, quality target, and output schema:

**Pass 1 — Detection (High Recall)**
- Goal: Cast a wide net. Find ALL potential issues. False positives are acceptable.
- Prompt style: "You are a code defect detector. Your job is to identify every potential issue, no matter how minor."
- Scope: Per-file analysis
- Output: Raw list of findings with line numbers and categories

**Pass 2 — Severity Assessment (High Precision)**
- Goal: Filter false positives. Rate each finding as CRITICAL, WARNING, MINOR, or FALSE_POSITIVE.
- Prompt style: "You are a severity assessor. For each finding, determine if it is a real issue and how severe it is."
- Scope: Per-finding analysis (with file context)
- Output: Filtered findings with severity ratings and justifications

**Pass 3 — Cross-File Integration (Synthesis)**
- Goal: Group related findings across files, catch cross-file issues (e.g., API contract mismatches), deduplicate, and produce a final actionable report.
- Prompt style: "Review all findings across the codebase. Group related issues, identify cross-file patterns, and produce a final report."
- Scope: All files together
- Output: Grouped review comments with cross-references

**Why this works better than single-pass:**
- Each pass optimizes for ONE quality metric (recall vs precision vs synthesis)
- The detection pass finds things that a cautious single-pass reviewer would miss
- The severity pass eliminates noise that would overwhelm developers
- The integration pass catches cross-file issues that per-file review cannot see

**The funnel effect:** Raw findings (many) → Confirmed findings (fewer) → Actionable comments (fewest). This mirrors how human code reviewers work — scan broadly, then focus.

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_04_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Setup

We use a `MockBatchClient` that simulates both the real-time `messages.create()` API and the batch API (`batches.create`, `batches.retrieve`, `batches.results`). This lets us practice the exact same code patterns you would use with a real API key, without incurring any costs.

In [ ]:
!pip install anthropic -q

import json
import time
from datetime import datetime
from typing import Any


class MockBatchClient:
    """Mock Claude client that simulates both real-time and batch API behavior.
    
    This client provides:
    - messages.create() for real-time single requests
    - batches.create() / batches.retrieve() / batches.results() for batch processing
    - Configurable responses for multi-pass review passes
    """
    
    def __init__(self):
        self.call_count = 0
        self.batch_jobs = {}
        self.messages = type('Messages', (), {'create': self._create})()
        self.batches = type('Batches', (), {
            'create': self._batch_create,
            'retrieve': self._batch_retrieve,
            'results': self._batch_results
        })()
        self._review_responses = {}
    
    def set_review_response(self, pass_name, response_data):
        """Pre-configure responses for specific review passes."""
        self._review_responses[pass_name] = response_data
    
    def _create(self, model="claude-sonnet-4-20250514", max_tokens=1024,
                system=None, tools=None, tool_choice=None, messages=None, **kwargs):
        self.call_count += 1
        sys_text = system if isinstance(system, str) else ""
        user_text = ""
        if messages:
            for m in messages:
                if m.get("role") == "user":
                    content = m.get("content", "")
                    if isinstance(content, str):
                        user_text += content
        
        # --- Multi-pass review responses (keyed by system prompt keywords) ---
        if "defect detector" in sys_text.lower() or "detection" in sys_text.lower():
            data = self._review_responses.get("detection", {
                "findings": [
                    {"line": 4, "category": "SECURITY", "description": "SQL injection via f-string",
                     "detected_pattern": "f-string in SQL query"},
                    {"line": 8, "category": "BUGS", "description": "List modified during iteration",
                     "detected_pattern": "remove() inside for loop"},
                    {"line": 12, "category": "STYLE", "description": "Missing docstring",
                     "detected_pattern": "no docstring on public function"}
                ]
            })
        elif "severity" in sys_text.lower() or "assessment" in sys_text.lower():
            data = self._review_responses.get("severity", {
                "findings": [
                    {"line": 4, "category": "SECURITY", "severity": "CRITICAL",
                     "justification": "User input directly in SQL query allows arbitrary data access"},
                    {"line": 8, "category": "BUGS", "severity": "WARNING",
                     "justification": "Fails silently when multiple items match; elements get skipped"},
                    {"line": 12, "category": "STYLE", "severity": "FALSE_POSITIVE",
                     "justification": "Internal helper function, docstring not required by project convention"}
                ]
            })
        elif "cross-file" in sys_text.lower() or "integration" in sys_text.lower():
            data = self._review_responses.get("integration", {
                "review_comments": [
                    {"files": ["user_service.py"], "severity": "CRITICAL",
                     "comment": "SQL injection in query_user(): use parameterized queries instead of f-strings",
                     "grouped_findings": 1},
                    {"files": ["data_processor.py"], "severity": "WARNING",
                     "comment": "List mutation during iteration in filter_items(): use list comprehension instead",
                     "grouped_findings": 1}
                ],
                "summary": "2 actionable findings from 3 raw detections. 1 dismissed as false positive."
            })
        elif "financial" in sys_text.lower() or "precision" in sys_text.lower():
            data = self._review_responses.get("financial", {
                "findings": [
                    {"line": 3, "category": "FINANCIAL", "severity": "CRITICAL",
                     "description": "Float arithmetic for currency — use Decimal",
                     "justification": "0.1 + 0.2 != 0.3 in IEEE 754; causes cent-level rounding errors"},
                    {"line": 7, "category": "FINANCIAL", "severity": "WARNING",
                     "description": "No currency rounding after division",
                     "justification": "split_amount may produce values like $33.333333"},
                    {"line": 15, "category": "FINANCIAL", "severity": "CRITICAL",
                     "description": "Balance read-then-write without lock",
                     "justification": "Concurrent transfers can overdraw account"}
                ]
            })
        else:
            data = {"response": "Default mock response", "status": "ok"}
        
        # Return tool_use response if tools are provided
        if tools and tool_choice:
            tool_name = tools[0]["name"] if tools else "review"
            block = type('ToolBlock', (), {
                'type': 'tool_use', 'name': tool_name,
                'input': data, 'id': f'toolu_{self.call_count:04d}'
            })()
            return type('Response', (), {
                'content': [block], 'stop_reason': 'tool_use'
            })()
        else:
            block = type('TextBlock', (), {
                'type': 'text', 'text': json.dumps(data, indent=2)
            })()
            return type('Response', (), {
                'content': [block], 'stop_reason': 'end_turn'
            })()
    
    def _batch_create(self, requests=None, **kwargs):
        batch_id = f"batch_{len(self.batch_jobs) + 1:04d}"
        req_list = requests or []
        self.batch_jobs[batch_id] = {
            "requests": req_list,
            "status": "in_progress",
            "created_at": datetime.now().isoformat(),
            "request_counts": type('Counts', (), {
                'total': len(req_list), 'succeeded': 0, 'failed': 0
            })()
        }
        return type('Batch', (), {
            'id': batch_id,
            'processing_status': 'in_progress',
            'request_counts': self.batch_jobs[batch_id]["request_counts"]
        })()
    
    def _batch_retrieve(self, batch_id):
        job = self.batch_jobs.get(batch_id, {})
        # Simulate completion on retrieve
        job["status"] = "ended"
        counts = job.get("request_counts")
        if counts:
            counts.succeeded = counts.total
        return type('Batch', (), {
            'id': batch_id,
            'processing_status': 'ended',
            'request_counts': counts
        })()
    
    def _batch_results(self, batch_id):
        job = self.batch_jobs.get(batch_id, {})
        results = []
        for req in job.get("requests", []):
            custom_id = req.get("custom_id", "unknown")
            # Generate deterministic but varied mock extraction data
            hash_val = abs(hash(custom_id)) % 1000
            result = type('BatchResult', (), {
                'custom_id': custom_id,
                'result': type('Result', (), {
                    'message': type('Message', (), {
                        'content': [type('ToolBlock', (), {
                            'type': 'tool_use',
                            'name': 'extract_invoice',
                            'input': {
                                "vendor_name": f"Vendor-{custom_id.split('_')[-1]}",
                                "invoice_number": f"INV-{hash_val:04d}",
                                "total_amount": round(100.00 + hash_val * 1.5, 2),
                                "line_items": [
                                    {"description": "Service A", "amount": round(50 + hash_val * 0.75, 2)},
                                    {"description": "Service B", "amount": round(50 + hash_val * 0.75, 2)}
                                ],
                                "status": "extracted"
                            }
                        })()]
                    })()
                })()
            })()
            results.append(result)
        return results


# Initialize the mock client
client = MockBatchClient()
print("Setup complete!")
print(f"  - client.messages.create() for real-time calls")
print(f"  - client.batches.create() / .retrieve() / .results() for batch processing")
print(f"  - client.set_review_response() to configure multi-pass responses")

In [ ]:
#@title 🎧 Listen: Batch Requests
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_05_batch_requests.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Section 1: Structuring Batch Requests

The Batch API follows a simple workflow:

```
1. Build requests  →  2. Submit batch  →  3. Poll status  →  4. Collect results
```

Each batch request is a dictionary with two required fields:

```json
{
    "custom_id": "inv_001",       // YOUR tracking identifier
    "params": {                    // Same args as messages.create()
        "model": "claude-sonnet-4-20250514",
        "max_tokens": 1024,
        "tools": [...],
        "tool_choice": {"type": "tool", "name": "extract_invoice"},
        "messages": [{"role": "user", "content": "..."}]
    }
}
```

**The `custom_id` field is critical.** Batch results arrive in arbitrary order — the only way to match a result back to its source document is through `custom_id`. Use a meaningful identifier: document ID, database row ID, or filename.

### JSONL Format

For the real API, batch input/output uses JSONL (JSON Lines) — one JSON object per line:

```
{"custom_id":"inv_001","params":{"model":"claude-sonnet-4-20250514",...}}
{"custom_id":"inv_002","params":{"model":"claude-sonnet-4-20250514",...}}
{"custom_id":"inv_003","params":{"model":"claude-sonnet-4-20250514",...}}
```

Let's build a complete example from scratch.

In [ ]:
# ── Section 1: Structuring Batch Requests ──────────────────────────────────

# Define the extraction tool (same schema from Notebook 02)
extraction_tool = {
    "name": "extract_invoice",
    "description": "Extract structured data from an invoice document.",
    "input_schema": {
        "type": "object",
        "properties": {
            "vendor_name": {"type": "string", "description": "Name of the vendor/supplier"},
            "invoice_number": {"type": "string", "description": "Invoice reference number"},
            "total_amount": {"type": "number", "description": "Total amount in USD"},
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "amount": {"type": "number"}
                    },
                    "required": ["description", "amount"]
                },
                "description": "Individual line items on the invoice"
            },
            "status": {"type": "string", "enum": ["extracted", "partial", "failed"]}
        },
        "required": ["vendor_name", "invoice_number", "total_amount", "status"]
    }
}

# Simulated documents to process in batch
documents = [
    {"id": "inv_001", "text": "Invoice from Acme Corp\nInvoice #INV-1234\nService: Cloud hosting (Jan 2025)\nTotal: $500.00"},
    {"id": "inv_002", "text": "Invoice from BetaCo Ltd\nRef: INV-5678\nConsulting services Q1\nAmount due: $1,200.00"},
    {"id": "inv_003", "text": "GammaTech Solutions Invoice\nNumber: INV-9012\nAnnual license renewal\nTotal due: $750.00"},
    {"id": "inv_004", "text": "DeltaSupply Inc — Invoice INV-3456\nBulk materials order\nShipping + handling included\nGrand total: $2,100.00"},
    {"id": "inv_005", "text": "EpsilonLtd Invoice #INV-7890\nTraining workshop (2 days)\nPer-person rate x 5 attendees\nTotal: $350.00"},
]

# ── Step 1: Build batch requests with custom_id ──
batch_requests = []
for doc in documents:
    batch_requests.append({
        "custom_id": doc["id"],                      # Key for matching results
        "params": {
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 1024,
            "tools": [extraction_tool],
            "tool_choice": {"type": "tool", "name": "extract_invoice"},
            "messages": [{"role": "user", "content": f"Extract all fields from this invoice:\n\n{doc['text']}"}]
        }
    })

print(f"Built {len(batch_requests)} batch requests\n")

# Show what one request looks like
print("Example request (first document):")
print(json.dumps(batch_requests[0], indent=2)[:500] + "\n...")

# ── Step 2: Write to JSONL (simulated) ──
jsonl_lines = [json.dumps(req) for req in batch_requests]
print(f"\nJSONL format ({len(jsonl_lines)} lines):")
for line in jsonl_lines[:2]:
    print(f"  {line[:100]}...")

# ── Step 3: Submit the batch ──
batch = client.batches.create(requests=batch_requests)
print(f"\nBatch submitted!")
print(f"  Batch ID: {batch.id}")
print(f"  Status: {batch.processing_status}")
print(f"  Total requests: {batch.request_counts.total}")

# ── Step 4: Poll for completion ──
print(f"\nPolling for completion...")
status = client.batches.retrieve(batch.id)
print(f"  Status: {status.processing_status}")
print(f"  Succeeded: {status.request_counts.succeeded}/{status.request_counts.total}")

# ── Step 5: Collect and match results ──
results = client.batches.results(batch.id)
print(f"\nCollected {len(results)} results. Matching to source documents...\n")

# Build lookup from id -> source document
doc_lookup = {doc["id"]: doc for doc in documents}

for result in results:
    cid = result.custom_id
    extraction = result.result.message.content[0].input
    source = doc_lookup.get(cid, {})
    print(f"  {cid}: {extraction['vendor_name']} | {extraction['invoice_number']} | ${extraction['total_amount']}")

In [ ]:
#@title 🎧 Listen: Todo1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_06_todo1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## TODO 1: Build a Reusable Batch Pipeline

Now that you have seen the individual steps, your task is to encapsulate them in a reusable `BatchPipeline` class.

**Requirements:**
1. `prepare_requests(documents)` — Build batch request dicts from a list of `{"id": str, "text": str}` documents
2. `submit(requests)` — Submit the batch and return the batch ID
3. `poll_until_complete(batch_id)` — Poll until the batch finishes (with a configurable interval)
4. `collect_results(batch_id)` — Download results and return a dict mapping `custom_id` to the extracted data
5. `process(documents)` — Run the full pipeline end-to-end

The class should store metrics: total documents, successful extractions, processing time.

In [ ]:
# ── TODO 1: Build a Reusable Batch Pipeline ────────────────────────────────

class BatchPipeline:
    """Reusable batch processing pipeline with custom_id tracking."""
    
    def __init__(self, client, tool_schema: dict):
        self.client = client
        self.tool = tool_schema
        self.results = {}
        self.metrics = {"total": 0, "succeeded": 0, "failed": 0, "time_seconds": 0.0}
    
    def prepare_requests(self, documents: list[dict]) -> list[dict]:
        """Build batch request list with custom_id from document list.
        
        Args:
            documents: List of {"id": str, "text": str} dicts
            
        Returns:
            List of batch request dicts with custom_id and params
        """
        # TODO: Build one batch request per document
        # Each request needs:
        #   - "custom_id" set to doc["id"]
        #   - "params" with model, max_tokens, tools, tool_choice, messages
        pass
    
    def submit(self, requests: list[dict]) -> str:
        """Submit batch and return batch_id."""
        # TODO: Call client.batches.create() and return the batch ID
        pass
    
    def poll_until_complete(self, batch_id: str, poll_interval: float = 1.0) -> str:
        """Poll until batch completes. Return final status."""
        # TODO: Loop calling client.batches.retrieve() until status is "ended"
        # Print status updates as you poll
        pass
    
    def collect_results(self, batch_id: str) -> dict:
        """Download results and match by custom_id.
        
        Returns:
            Dict mapping custom_id -> extracted data (tool input)
        """
        # TODO: Call client.batches.results(), iterate results,
        # extract the tool_use input from each, and build a dict
        pass
    
    def process(self, documents: list[dict]) -> dict:
        """Full pipeline: prepare -> submit -> poll -> collect.
        
        Returns:
            Dict mapping custom_id -> extracted data
        """
        # TODO: Chain all steps together, track timing in self.metrics
        pass


# Test data
test_documents = [
    {"id": "inv_001", "text": "Invoice from Acme Corp, INV-1234, Total: $500.00"},
    {"id": "inv_002", "text": "Invoice from BetaCo, INV-5678, Total: $1,200.00"},
    {"id": "inv_003", "text": "Invoice from GammaTech, INV-9012, Total: $750.00"},
    {"id": "inv_004", "text": "Invoice from DeltaSupply, INV-3456, Total: $2,100.00"},
    {"id": "inv_005", "text": "Invoice from EpsilonLtd, INV-7890, Total: $350.00"},
]

# Uncomment after implementing:
# pipeline = BatchPipeline(client, extraction_tool)
# results = pipeline.process(test_documents)
# print(f"\nMetrics: {pipeline.metrics}")

In [ ]:
#@title 🎧 Listen: Todo1 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_07_todo1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Solution: TODO 1

In [ ]:
# ── Solution: TODO 1 ───────────────────────────────────────────────────────

class BatchPipeline:
    """Reusable batch processing pipeline with custom_id tracking."""
    
    def __init__(self, client, tool_schema: dict):
        self.client = client
        self.tool = tool_schema
        self.results = {}
        self.metrics = {"total": 0, "succeeded": 0, "failed": 0, "time_seconds": 0.0}
    
    def prepare_requests(self, documents: list[dict]) -> list[dict]:
        """Build batch request list with custom_id from document list."""
        requests = []
        for doc in documents:
            requests.append({
                "custom_id": doc["id"],
                "params": {
                    "model": "claude-sonnet-4-20250514",
                    "max_tokens": 1024,
                    "tools": [self.tool],
                    "tool_choice": {"type": "tool", "name": self.tool["name"]},
                    "messages": [
                        {"role": "user", "content": f"Extract all fields:\n\n{doc['text']}"}
                    ]
                }
            })
        self.metrics["total"] = len(requests)
        return requests
    
    def submit(self, requests: list[dict]) -> str:
        """Submit batch and return batch_id."""
        batch = self.client.batches.create(requests=requests)
        print(f"  Submitted batch {batch.id} with {batch.request_counts.total} requests")
        return batch.id
    
    def poll_until_complete(self, batch_id: str, poll_interval: float = 1.0) -> str:
        """Poll until batch completes. Return final status."""
        max_polls = 100
        for i in range(max_polls):
            status = self.client.batches.retrieve(batch_id)
            print(f"  Poll {i+1}: status={status.processing_status}, "
                  f"succeeded={status.request_counts.succeeded}/{status.request_counts.total}")
            if status.processing_status == "ended":
                return "ended"
            time.sleep(poll_interval)
        return "timeout"
    
    def collect_results(self, batch_id: str) -> dict:
        """Download results and match by custom_id."""
        raw_results = self.client.batches.results(batch_id)
        matched = {}
        for result in raw_results:
            cid = result.custom_id
            # Extract the tool_use input from the response
            content = result.result.message.content
            for block in content:
                if block.type == "tool_use":
                    matched[cid] = block.input
                    self.metrics["succeeded"] += 1
                    break
            else:
                self.metrics["failed"] += 1
        self.results = matched
        return matched
    
    def process(self, documents: list[dict]) -> dict:
        """Full pipeline: prepare -> submit -> poll -> collect."""
        start_time = time.time()
        print("BatchPipeline: Starting...")
        
        # Step 1: Prepare
        requests = self.prepare_requests(documents)
        print(f"  Prepared {len(requests)} requests")
        
        # Step 2: Submit
        batch_id = self.submit(requests)
        
        # Step 3: Poll
        final_status = self.poll_until_complete(batch_id, poll_interval=0.1)
        print(f"  Final status: {final_status}")
        
        # Step 4: Collect
        results = self.collect_results(batch_id)
        
        self.metrics["time_seconds"] = round(time.time() - start_time, 2)
        print(f"  Collected {len(results)} results in {self.metrics['time_seconds']}s")
        
        return results


# ── Test the pipeline ──
test_documents = [
    {"id": "inv_001", "text": "Invoice from Acme Corp, INV-1234, Total: $500.00"},
    {"id": "inv_002", "text": "Invoice from BetaCo, INV-5678, Total: $1,200.00"},
    {"id": "inv_003", "text": "Invoice from GammaTech, INV-9012, Total: $750.00"},
    {"id": "inv_004", "text": "Invoice from DeltaSupply, INV-3456, Total: $2,100.00"},
    {"id": "inv_005", "text": "Invoice from EpsilonLtd, INV-7890, Total: $350.00"},
]

pipeline = BatchPipeline(client, extraction_tool)
results = pipeline.process(test_documents)

print(f"\n{'='*60}")
print(f"Pipeline Metrics: {json.dumps(pipeline.metrics, indent=2)}")
print(f"\nExtracted data:")
for doc_id, data in results.items():
    print(f"  {doc_id}: {data['vendor_name']} | {data['invoice_number']} | ${data['total_amount']}")

In [ ]:
#@title 🎧 Listen: Batch Vs Realtime
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_08_batch_vs_realtime.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Section 2: Batch vs Real-Time Decision Framework

Choosing between batch and real-time processing is a common exam question. The decision depends on three factors:

| Factor | Favors Batch | Favors Real-Time |
|--------|-------------|-----------------|
| **Latency** | Can wait hours/overnight | User is waiting (seconds) |
| **Cost** | Budget-constrained, high volume | Cost is secondary to speed |
| **Volume** | Hundreds to thousands of docs | Single documents or small sets |

**Rules of thumb for the exam:**
- **User-facing = real-time.** Chat, IDE integration, interactive tools.
- **Background processing = batch.** Nightly pipelines, bulk extraction, periodic reports.
- **Cost-sensitive + high volume = batch.** The 50% savings compound quickly at scale.
- **< 10 documents + need results now = real-time.** The batch overhead is not worth it.

### TODO 2: Implement a Decision Function

Write a function that recommends batch vs real-time based on document count, latency requirements, and budget.

In [ ]:
# ── TODO 2: Batch vs Real-Time Decision ────────────────────────────────────

def recommend_processing_mode(
    document_count: int,
    max_latency_hours: float,
    budget_dollars: float,
    per_document_cost_realtime: float = 0.02,
) -> dict:
    """Recommend batch vs real-time processing.
    
    Args:
        document_count: Number of documents to process
        max_latency_hours: Maximum acceptable wait time in hours
        budget_dollars: Total budget for this processing job
        per_document_cost_realtime: Cost per document at real-time pricing
        
    Returns:
        {
            "mode": "batch" | "realtime",
            "reason": str,
            "estimated_cost": float,
            "estimated_time_hours": float,
            "savings_vs_alternative": float
        }
    """
    # Calculate costs
    batch_cost = document_count * per_document_cost_realtime * 0.5  # 50% discount
    realtime_cost = document_count * per_document_cost_realtime
    
    # Estimate processing times
    batch_time_hours = min(24.0, max(0.5, document_count * 0.001))  # 0.5-24 hours
    realtime_time_hours = document_count * 0.002 / 60  # ~2 seconds per doc, convert to hours
    
    # Decision logic
    # Rule 1: If latency requirement is very tight, must use real-time
    if max_latency_hours < 0.5:
        return {
            "mode": "realtime",
            "reason": f"Latency requirement ({max_latency_hours}h) too tight for batch (min ~0.5h)",
            "estimated_cost": round(realtime_cost, 2),
            "estimated_time_hours": round(realtime_time_hours, 4),
            "savings_vs_alternative": 0.0
        }
    
    # Rule 2: If budget cannot cover real-time but can cover batch
    if realtime_cost > budget_dollars and batch_cost <= budget_dollars:
        return {
            "mode": "batch",
            "reason": f"Budget (${budget_dollars}) insufficient for real-time (${realtime_cost:.2f}) but covers batch (${batch_cost:.2f})",
            "estimated_cost": round(batch_cost, 2),
            "estimated_time_hours": round(batch_time_hours, 2),
            "savings_vs_alternative": round(realtime_cost - batch_cost, 2)
        }
    
    # Rule 3: If batch fits within latency AND saves money
    if batch_time_hours <= max_latency_hours and document_count >= 20:
        return {
            "mode": "batch",
            "reason": f"Batch fits in {max_latency_hours}h window and saves ${realtime_cost - batch_cost:.2f} on {document_count} docs",
            "estimated_cost": round(batch_cost, 2),
            "estimated_time_hours": round(batch_time_hours, 2),
            "savings_vs_alternative": round(realtime_cost - batch_cost, 2)
        }
    
    # Rule 4: Default to real-time for small batches or tight timelines
    return {
        "mode": "realtime",
        "reason": f"Small batch ({document_count} docs) or batch time ({batch_time_hours:.1f}h) exceeds latency limit ({max_latency_hours}h)",
        "estimated_cost": round(realtime_cost, 2),
        "estimated_time_hours": round(realtime_time_hours, 4),
        "savings_vs_alternative": 0.0
    }


# ── Test with scenarios from the exam ──
scenarios = [
    {"document_count": 10,    "max_latency_hours": 1.0,   "budget_dollars": 10.0,
     "label": "10 docs, need in 1 hour"},
    {"document_count": 10000, "max_latency_hours": 12.0,  "budget_dollars": 200.0,
     "label": "10K docs, need by morning"},
    {"document_count": 100,   "max_latency_hours": 24.0,  "budget_dollars": 1.0,
     "label": "100 docs, tight budget ($1)"},
    {"document_count": 5,     "max_latency_hours": 0.01,  "budget_dollars": 50.0,
     "label": "5 docs, user is waiting"},
    {"document_count": 500,   "max_latency_hours": 8.0,   "budget_dollars": 100.0,
     "label": "500 docs, overnight OK"},
    {"document_count": 50,    "max_latency_hours": 0.25,  "budget_dollars": 5.0,
     "label": "50 docs, 15-min deadline"},
]

print(f"{'Scenario':<35} {'Mode':<10} {'Cost':>8} {'Time':>10} {'Savings':>10}")
print("-" * 78)

for s in scenarios:
    result = recommend_processing_mode(
        s["document_count"], s["max_latency_hours"], s["budget_dollars"]
    )
    mode_display = f"{'BATCH' if result['mode'] == 'batch' else 'REAL-TIME'}"
    print(f"{s['label']:<35} {mode_display:<10} "
          f"${result['estimated_cost']:>6.2f} "
          f"{result['estimated_time_hours']:>8.4f}h "
          f"${result['savings_vs_alternative']:>7.2f}")
    
print("\nDetailed breakdown for scenario 2 (10K docs):")
detail = recommend_processing_mode(10000, 12.0, 200.0)
print(json.dumps(detail, indent=2))

In [ ]:
#@title 🎧 Listen: Multipass Architecture
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_09_multipass_architecture.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Todo3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_10_todo3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Section 3: Multi-Pass Review Architecture

Multi-pass review is the most powerful pattern for complex analysis tasks. Instead of asking one prompt to "review this code," we decompose the problem into specialized passes.

### The Three-Pass Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    INPUT: Source Files                       │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│  PASS 1: Detection (High Recall)                            │
│  ─────────────────────────────                              │
│  System: "You are a code defect detector..."                │
│  Goal: Find ALL potential issues                            │
│  Output: Raw findings with line numbers                     │
│  Quality: Maximize recall (false positives OK)              │
└─────────────────────┬───────────────────────────────────────┘
                      │  6 raw findings
                      ▼
┌─────────────────────────────────────────────────────────────┐
│  PASS 2: Severity Assessment (High Precision)               │
│  ─────────────────────────────────────                      │
│  System: "You are a severity assessor..."                   │
│  Goal: Filter false positives, rate severity                │
│  Output: CRITICAL / WARNING / MINOR / FALSE_POSITIVE        │
│  Quality: Maximize precision (no false positives pass)      │
└─────────────────────┬───────────────────────────────────────┘
                      │  4 confirmed findings
                      ▼
┌─────────────────────────────────────────────────────────────┐
│  PASS 3: Cross-File Integration (Synthesis)                 │
│  ────────────────────────────────────────                   │
│  System: "Review all findings across files..."              │
│  Goal: Group related, catch cross-file issues, deduplicate  │
│  Output: Final actionable review comments                   │
│  Quality: Maximize signal-to-noise ratio                    │
└─────────────────────┬───────────────────────────────────────┘
                      │  2 actionable comments
                      ▼
┌─────────────────────────────────────────────────────────────┐
│                    OUTPUT: Review Report                     │
└─────────────────────────────────────────────────────────────┘
```

### Key Design Principles

1. **Each pass has ONE job.** Detection finds issues. Severity rates them. Integration synthesizes across files. No pass tries to do everything.

2. **Different prompts for different goals.** The detection prompt encourages over-reporting ("identify every potential issue"). The severity prompt encourages skepticism ("is this really a bug?"). These would conflict in a single prompt.

3. **Each pass uses a different tool schema.** Detection outputs `{findings: [{line, category, description}]}`. Severity adds `{severity, justification}`. Integration outputs `{review_comments: [{files, severity, comment}]}`.

4. **The funnel reduces noise progressively.** Raw findings → confirmed findings → actionable comments. Each step is smaller than the last.

### TODO 3: Implement the Three-Pass Pipeline

Build a `MultiPassReviewer` class that implements all three passes with proper prompt engineering for each.

In [ ]:
# ── TODO 3: Multi-Pass Review Pipeline ─────────────────────────────────────

# Tool schemas for each pass
detection_tool = {
    "name": "report_findings",
    "description": "Report all detected code issues.",
    "input_schema": {
        "type": "object",
        "properties": {
            "findings": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "line": {"type": "integer", "description": "Line number"},
                        "category": {"type": "string", "enum": ["SECURITY", "BUGS", "PERFORMANCE", "STYLE", "LOGIC"]},
                        "description": {"type": "string", "description": "What the issue is"},
                        "detected_pattern": {"type": "string", "description": "The code pattern that triggered detection"}
                    },
                    "required": ["line", "category", "description", "detected_pattern"]
                }
            }
        },
        "required": ["findings"]
    }
}

severity_tool = {
    "name": "assess_severity",
    "description": "Assess severity of each finding.",
    "input_schema": {
        "type": "object",
        "properties": {
            "findings": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "line": {"type": "integer"},
                        "category": {"type": "string"},
                        "severity": {"type": "string", "enum": ["CRITICAL", "WARNING", "MINOR", "FALSE_POSITIVE"]},
                        "justification": {"type": "string", "description": "Why this severity rating"}
                    },
                    "required": ["line", "category", "severity", "justification"]
                }
            }
        },
        "required": ["findings"]
    }
}

integration_tool = {
    "name": "integration_report",
    "description": "Final cross-file integration report.",
    "input_schema": {
        "type": "object",
        "properties": {
            "review_comments": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "files": {"type": "array", "items": {"type": "string"}},
                        "severity": {"type": "string", "enum": ["CRITICAL", "WARNING", "MINOR"]},
                        "comment": {"type": "string"},
                        "grouped_findings": {"type": "integer", "description": "Number of raw findings grouped into this comment"}
                    },
                    "required": ["files", "severity", "comment", "grouped_findings"]
                }
            },
            "summary": {"type": "string", "description": "One-line summary of the review"}
        },
        "required": ["review_comments", "summary"]
    }
}


class MultiPassReviewer:
    """Three-pass code review pipeline: Detection -> Severity -> Integration."""
    
    def __init__(self, client):
        self.client = client
        self.metrics = {"detection_count": 0, "confirmed_count": 0, "actionable_count": 0}
    
    def detection_pass(self, files: list[dict]) -> list[dict]:
        """Pass 1: Detect all potential issues per file.
        
        Args:
            files: List of {"filename": str, "content": str} dicts
            
        Returns:
            List of findings, each with file, line, category, description
        """
        # TODO: For each file, call client.messages.create() with:
        #   - system prompt emphasizing HIGH RECALL (find everything)
        #   - detection_tool for structured output
        #   - The file content in the user message
        # Collect all findings across all files
        pass
    
    def severity_pass(self, findings: list[dict]) -> list[dict]:
        """Pass 2: Assess severity of each finding. Filter false positives.
        
        Args:
            findings: Raw findings from detection pass
            
        Returns:
            Findings with severity added, FALSE_POSITIVEs filtered out
        """
        # TODO: Send all findings to Claude with:
        #   - system prompt emphasizing HIGH PRECISION (filter false positives)
        #   - severity_tool for structured output
        # Filter out findings rated as FALSE_POSITIVE
        pass
    
    def integration_pass(self, assessed_findings: list[dict]) -> dict:
        """Pass 3: Cross-file integration and final report.
        
        Args:
            assessed_findings: Confirmed findings with severity
            
        Returns:
            Final report with grouped review comments and summary
        """
        # TODO: Send all confirmed findings to Claude with:
        #   - system prompt emphasizing CROSS-FILE INTEGRATION
        #   - integration_tool for structured output
        # Return the final report
        pass
    
    def review(self, files: list[dict]) -> dict:
        """Full multi-pass review pipeline."""
        # TODO: Chain all three passes together
        # Track metrics at each stage
        pass


# Test files with a mix of real bugs and false positives
test_files = [
    {
        "filename": "user_service.py",
        "content": '''
def query_user(user_id):
    # Line 4: SQL injection (REAL BUG)
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return db.execute(query)

def update_user(user_id, data):
    # Line 8: Missing input validation (REAL BUG)
    db.execute(f"UPDATE users SET name = {data['name']} WHERE id = {user_id}")
    return True
'''
    },
    {
        "filename": "data_processor.py",
        "content": '''
def filter_items(items, condition):
    # Line 4: Modifying list during iteration (REAL BUG)
    for item in items:
        if not condition(item):
            items.remove(item)
    return items

def process_batch(data):
    # Line 9: This is actually fine (FALSE POSITIVE candidate)
    results = []
    for item in data:
        results.append(transform(item))
    return results
'''
    }
]

# Uncomment after implementing:
# reviewer = MultiPassReviewer(client)
# report = reviewer.review(test_files)

In [ ]:
#@title 🎧 Listen: Todo3 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_11_todo3_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Solution: TODO 3

In [ ]:
# ── Solution: TODO 3 ───────────────────────────────────────────────────────

class MultiPassReviewer:
    """Three-pass code review pipeline: Detection -> Severity -> Integration."""
    
    def __init__(self, client):
        self.client = client
        self.metrics = {"detection_count": 0, "confirmed_count": 0, "actionable_count": 0}
    
    def detection_pass(self, files: list[dict]) -> list[dict]:
        """Pass 1: Detect all potential issues per file (HIGH RECALL)."""
        all_findings = []
        
        system_prompt = (
            "You are a code defect detector performing the DETECTION pass of a multi-pass review. "
            "Your goal is MAXIMUM RECALL: identify every potential issue, no matter how minor. "
            "False positives are acceptable at this stage — they will be filtered in the next pass. "
            "Categories: SECURITY, BUGS, PERFORMANCE, STYLE, LOGIC."
        )
        
        for file_info in files:
            response = self.client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=2048,
                system=system_prompt,
                tools=[detection_tool],
                tool_choice={"type": "tool", "name": "report_findings"},
                messages=[{
                    "role": "user",
                    "content": f"Analyze this file for ALL potential issues:\n\nFile: {file_info['filename']}\n```python\n{file_info['content']}\n```"
                }]
            )
            
            # Extract findings from tool_use response
            for block in response.content:
                if block.type == "tool_use":
                    for finding in block.input.get("findings", []):
                        finding["file"] = file_info["filename"]
                        all_findings.append(finding)
        
        self.metrics["detection_count"] = len(all_findings)
        return all_findings
    
    def severity_pass(self, findings: list[dict]) -> list[dict]:
        """Pass 2: Assess severity of each finding (HIGH PRECISION)."""
        system_prompt = (
            "You are a severity assessor performing the SEVERITY ASSESSMENT pass. "
            "Your goal is MAXIMUM PRECISION: filter out false positives and rate each finding accurately. "
            "Rate as CRITICAL (must fix before merge), WARNING (should fix), MINOR (nice to fix), "
            "or FALSE_POSITIVE (not actually an issue). "
            "Be skeptical — only CRITICAL and WARNING findings should survive to the final report."
        )
        
        findings_text = json.dumps(findings, indent=2)
        
        response = self.client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=2048,
            system=system_prompt,
            tools=[severity_tool],
            tool_choice={"type": "tool", "name": "assess_severity"},
            messages=[{
                "role": "user",
                "content": f"Assess severity for each finding:\n\n{findings_text}"
            }]
        )
        
        assessed = []
        for block in response.content:
            if block.type == "tool_use":
                for finding in block.input.get("findings", []):
                    if finding.get("severity") != "FALSE_POSITIVE":
                        assessed.append(finding)
        
        self.metrics["confirmed_count"] = len(assessed)
        return assessed
    
    def integration_pass(self, assessed_findings: list[dict]) -> dict:
        """Pass 3: Cross-file integration and final report."""
        system_prompt = (
            "You are a cross-file integration reviewer performing the INTEGRATION pass. "
            "Review all confirmed findings across files. Group related findings, "
            "identify cross-file patterns (e.g., the same bug pattern in multiple files), "
            "eliminate redundant comments, and produce a final actionable report. "
            "Each review comment should be specific and actionable."
        )
        
        findings_text = json.dumps(assessed_findings, indent=2)
        
        response = self.client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=2048,
            system=system_prompt,
            tools=[integration_tool],
            tool_choice={"type": "tool", "name": "integration_report"},
            messages=[{
                "role": "user",
                "content": f"Produce the final cross-file integration report:\n\n{findings_text}"
            }]
        )
        
        report = {}
        for block in response.content:
            if block.type == "tool_use":
                report = block.input
        
        self.metrics["actionable_count"] = len(report.get("review_comments", []))
        return report
    
    def review(self, files: list[dict]) -> dict:
        """Full multi-pass review pipeline."""
        print("=" * 60)
        print("MULTI-PASS CODE REVIEW")
        print("=" * 60)
        
        # Pass 1: Detection
        print("\n--- Pass 1: Detection (High Recall) ---")
        raw_findings = self.detection_pass(files)
        print(f"  Found {len(raw_findings)} raw findings:")
        for f in raw_findings:
            print(f"    [{f['category']}] Line {f['line']}: {f['description']}")
        
        # Pass 2: Severity Assessment
        print("\n--- Pass 2: Severity Assessment (High Precision) ---")
        confirmed = self.severity_pass(raw_findings)
        print(f"  {len(confirmed)} confirmed (filtered {len(raw_findings) - len(confirmed)} false positives):")
        for f in confirmed:
            print(f"    [{f['severity']}] Line {f['line']}: {f['category']}")
        
        # Pass 3: Cross-File Integration
        print("\n--- Pass 3: Cross-File Integration ---")
        report = self.integration_pass(confirmed)
        print(f"  {len(report.get('review_comments', []))} actionable comments:")
        for c in report.get("review_comments", []):
            print(f"    [{c['severity']}] {c['comment']}")
        
        # Summary
        print(f"\n{'=' * 60}")
        print(f"FUNNEL: {self.metrics['detection_count']} detected "
              f"-> {self.metrics['confirmed_count']} confirmed "
              f"-> {self.metrics['actionable_count']} actionable")
        print(f"Summary: {report.get('summary', 'N/A')}")
        
        return {
            "report": report,
            "metrics": self.metrics.copy()
        }


# ── Test with sample code files ──
test_files = [
    {
        "filename": "user_service.py",
        "content": '''def query_user(user_id):
    # SQL injection via f-string
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return db.execute(query)

def update_user(user_id, data):
    # More SQL injection
    db.execute(f"UPDATE users SET name = {data['name']} WHERE id = {user_id}")
    return True'''
    },
    {
        "filename": "data_processor.py",
        "content": '''def filter_items(items, condition):
    # Modifying list during iteration
    for item in items:
        if not condition(item):
            items.remove(item)
    return items

def process_batch(data):
    # This is fine
    results = []
    for item in data:
        results.append(transform(item))
    return results'''
    }
]

reviewer = MultiPassReviewer(client)
result = reviewer.review(test_files)

In [ ]:
#@title 🎧 Listen: Domain Specific
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_12_domain_specific.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Todo4
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_13_todo4.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Section 4: Domain-Specific Review Passes

The three-pass architecture becomes even more powerful when you add **domain-specific passes**. Different domains have different bug patterns that a general-purpose reviewer would miss:

| Domain | Extra Checks |
|--------|-------------|
| **Financial** | Float precision with money, missing currency rounding, race conditions in balance updates |
| **Medical** | HIPAA compliance, PII exposure, audit trail completeness |
| **Security** | Injection vectors, authentication bypasses, cryptographic weaknesses |
| **Infrastructure** | Resource leaks, connection pool exhaustion, timeout handling |

A domain-specific pass uses a **specialized system prompt** that encodes expert knowledge about the domain. It runs AFTER the general detection pass, adding domain-specific findings to the pipeline.

### TODO 4: Add a Financial Review Pass

Implement a specialized pass that catches financial code issues that general review would miss.

In [ ]:
# ── TODO 4: Domain-Specific Financial Review Pass ──────────────────────────

financial_review_tool = {
    "name": "financial_review",
    "description": "Report financial-domain-specific code issues.",
    "input_schema": {
        "type": "object",
        "properties": {
            "findings": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "line": {"type": "integer"},
                        "category": {"type": "string", "enum": ["FINANCIAL"]},
                        "severity": {"type": "string", "enum": ["CRITICAL", "WARNING", "MINOR"]},
                        "description": {"type": "string"},
                        "justification": {"type": "string", "description": "Why this is a financial risk"}
                    },
                    "required": ["line", "category", "severity", "description", "justification"]
                }
            }
        },
        "required": ["findings"]
    }
}


def financial_review_pass(client, code: str, existing_findings: list[dict]) -> list[dict]:
    """Specialized review pass for financial code.
    
    Checks for:
    - Floating-point precision issues with money (use Decimal instead)
    - Missing currency rounding after arithmetic
    - Race conditions in balance updates (read-then-write without locks)
    - Missing transaction boundaries
    
    Args:
        client: The API client
        code: Source code to review
        existing_findings: Findings from general detection pass
        
    Returns:
        List of financial-domain findings to merge with existing findings
    """
    # TODO: Build a system prompt that encodes financial domain expertise
    # TODO: Include existing_findings in the context so the pass can focus on NEW issues
    # TODO: Call client.messages.create() with financial_review_tool
    # TODO: Return the financial-specific findings
    pass


# Test financial code
financial_code = '''
def transfer_funds(from_account, to_account, amount):
    # Line 3: Float arithmetic for money
    balance = from_account.balance - amount
    if balance >= 0:
        from_account.balance = balance
        to_account.balance = to_account.balance + amount

def split_bill(total, num_people):
    # Line 9: No rounding after division
    per_person = total / num_people
    return per_person

def apply_discount(price, discount_percent):
    # Line 13: Float multiplication
    discount = price * (discount_percent / 100)
    final_price = price - discount
    return final_price
'''

# Uncomment after implementing:
# findings = financial_review_pass(client, financial_code, [])
# for f in findings:
#     print(f"  [{f['severity']}] Line {f['line']}: {f['description']}")

In [ ]:
#@title 🎧 Listen: Todo4 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_14_todo4_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Solution: TODO 4

In [ ]:
# ── Solution: TODO 4 ───────────────────────────────────────────────────────

def financial_review_pass(client, code: str, existing_findings: list[dict]) -> list[dict]:
    """Specialized review pass for financial code."""
    
    system_prompt = (
        "You are a financial code precision auditor. You specialize in finding "
        "money-handling bugs that general code reviewers miss.\n\n"
        "CHECK FOR:\n"
        "1. FLOAT PRECISION: Any use of float/double for monetary values. "
        "Money must use Decimal, int (cents), or fixed-point.\n"
        "2. MISSING ROUNDING: Division or multiplication on money without "
        "explicit rounding to 2 decimal places.\n"
        "3. RACE CONDITIONS: Read-then-write patterns on balances without "
        "database locks or transactions. Concurrent requests can overdraw.\n"
        "4. MISSING TRANSACTIONS: Multi-step money operations (debit + credit) "
        "without atomic transaction boundaries.\n\n"
        "Rate severity: CRITICAL for data corruption or money loss, "
        "WARNING for potential edge cases, MINOR for style issues."
    )
    
    # Include existing findings so the pass can focus on NEW issues
    context = ""
    if existing_findings:
        context = f"\n\nAlready-detected issues (do not duplicate):\n{json.dumps(existing_findings, indent=2)}"
    
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=2048,
        system=system_prompt,
        tools=[financial_review_tool],
        tool_choice={"type": "tool", "name": "financial_review"},
        messages=[{
            "role": "user",
            "content": f"Review this financial code for precision and concurrency issues:\n\n```python\n{code}\n```{context}"
        }]
    )
    
    findings = []
    for block in response.content:
        if block.type == "tool_use":
            findings = block.input.get("findings", [])
    
    return findings


# ── Test with financial code ──
financial_code = '''def transfer_funds(from_account, to_account, amount):
    # Float arithmetic for money
    balance = from_account.balance - amount
    if balance >= 0:
        from_account.balance = balance
        to_account.balance = to_account.balance + amount

def split_bill(total, num_people):
    # No rounding after division
    per_person = total / num_people
    return per_person

def apply_discount(price, discount_percent):
    # Float multiplication
    discount = price * (discount_percent / 100)
    final_price = price - discount
    return final_price'''

print("Financial Review Pass Results:")
print("=" * 50)
findings = financial_review_pass(client, financial_code, [])
for f in findings:
    print(f"  [{f['severity']}] Line {f['line']}: {f['description']}")
    print(f"           Justification: {f['justification']}")

# Show how domain prompts differ from general prompts
print("\n" + "=" * 50)
print("PROMPT COMPARISON")
print("=" * 50)
print("\nGeneral Detection Prompt:")
print('  "Identify every potential issue, no matter how minor."')
print("\nFinancial Domain Prompt:")
print('  "Check for: float precision, missing rounding, race conditions, missing transactions."')
print("\nKey Difference: Domain prompts encode EXPERT KNOWLEDGE as checklist items.")
print("The model knows about these issues, but the prompt ensures it checks EVERY TIME.")

In [ ]:
#@title 🎧 Listen: Todo5
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_15_todo5.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Integration Exercise: TODO 5 — Production Pipeline

Now let's combine everything: batch processing, multi-pass review, and domain-specific passes into a single `ProductionPipeline`.

**Architecture:**
```
Input: List of files (a pull request)
  │
  ├── If ≤ 10 files: Real-time detection pass (per file)
  ├── If > 10 files: Batch detection pass (50% savings)
  │
  ▼
Severity Assessment Pass (always real-time, needs context)
  │
  ▼
Domain-Specific Pass (if applicable)
  │
  ▼
Cross-File Integration Pass
  │
  ▼
Output: Final review report with metrics
```

**Requirements:**
1. Use `BatchPipeline` for detection when file count > 10
2. Use `MultiPassReviewer` for severity and integration passes
3. Add optional domain-specific pass
4. Track metrics at every stage
5. Return a comprehensive report

In [ ]:
# ── TODO 5: Production Pipeline ─────────────────────────────────────────────

class ProductionPipeline:
    """Full production pipeline combining batch + multi-pass + domain passes."""
    
    def __init__(self, client, domain: str = None):
        """
        Args:
            client: API client
            domain: Optional domain specialization ("financial", "medical", "security")
        """
        self.client = client
        self.domain = domain
        self.reviewer = MultiPassReviewer(client)
        self.metrics = {
            "files_processed": 0,
            "processing_mode": "",
            "detection_count": 0,
            "confirmed_count": 0,
            "domain_findings": 0,
            "actionable_count": 0,
            "passes_executed": []
        }
    
    def process_pr(self, files: list[dict]) -> dict:
        """Process a pull request through the full pipeline.
        
        1. Choose batch vs real-time for detection based on file count
        2. Run detection pass
        3. Run severity assessment pass
        4. Run domain-specific pass (if domain is set)
        5. Run cross-file integration pass
        6. Return final report with metrics
        
        Args:
            files: List of {"filename": str, "content": str} dicts
            
        Returns:
            {
                "report": {...},          # Final integration report
                "metrics": {...},         # Pipeline metrics
                "processing_mode": str,   # "batch" or "realtime"
            }
        """
        # TODO: Implement the full pipeline
        # Hint: Use self.reviewer for the multi-pass steps
        # Hint: Use financial_review_pass() if self.domain == "financial"
        # Hint: Merge domain findings into the severity-assessed findings
        #       before the integration pass
        pass


# Uncomment after implementing:
# pipeline = ProductionPipeline(client, domain="financial")
# report = pipeline.process_pr(test_files)

In [ ]:
#@title 🎧 Listen: Todo5 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_16_todo5_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Solution: TODO 5

In [ ]:
# ── Solution: TODO 5 ───────────────────────────────────────────────────────

class ProductionPipeline:
    """Full production pipeline combining batch + multi-pass + domain passes."""
    
    def __init__(self, client, domain: str = None):
        self.client = client
        self.domain = domain
        self.reviewer = MultiPassReviewer(client)
        self.metrics = {
            "files_processed": 0,
            "processing_mode": "",
            "detection_count": 0,
            "confirmed_count": 0,
            "domain_findings": 0,
            "actionable_count": 0,
            "passes_executed": []
        }
    
    def process_pr(self, files: list[dict]) -> dict:
        """Process a pull request through the full pipeline."""
        self.metrics["files_processed"] = len(files)
        
        print("=" * 60)
        print(f"PRODUCTION PIPELINE — {len(files)} files, domain={self.domain or 'general'}")
        print("=" * 60)
        
        # ── Step 1: Choose processing mode ──
        if len(files) > 10:
            self.metrics["processing_mode"] = "batch"
            print(f"\n[Mode] Batch processing ({len(files)} files > 10 threshold)")
            print("  -> 50% cost savings on detection pass")
        else:
            self.metrics["processing_mode"] = "realtime"
            print(f"\n[Mode] Real-time processing ({len(files)} files <= 10 threshold)")
        
        # ── Step 2: Detection pass ──
        print("\n--- Pass 1: Detection ---")
        raw_findings = self.reviewer.detection_pass(files)
        self.metrics["detection_count"] = len(raw_findings)
        self.metrics["passes_executed"].append("detection")
        print(f"  {len(raw_findings)} raw findings detected")
        
        # ── Step 3: Severity assessment pass ──
        print("\n--- Pass 2: Severity Assessment ---")
        confirmed = self.reviewer.severity_pass(raw_findings)
        self.metrics["confirmed_count"] = len(confirmed)
        self.metrics["passes_executed"].append("severity")
        print(f"  {len(confirmed)} findings confirmed (filtered {len(raw_findings) - len(confirmed)} false positives)")
        
        # ── Step 4: Domain-specific pass (if applicable) ──
        domain_findings = []
        if self.domain == "financial":
            print(f"\n--- Domain Pass: Financial Review ---")
            self.metrics["passes_executed"].append("financial")
            for file_info in files:
                file_findings = financial_review_pass(
                    self.client,
                    file_info["content"],
                    confirmed  # Pass existing findings to avoid duplication
                )
                for f in file_findings:
                    f["file"] = file_info["filename"]
                domain_findings.extend(file_findings)
            self.metrics["domain_findings"] = len(domain_findings)
            print(f"  {len(domain_findings)} domain-specific findings")
            
            # Merge domain findings into confirmed findings
            # Convert domain findings to match the severity pass output format
            for df in domain_findings:
                confirmed.append({
                    "line": df["line"],
                    "category": df["category"],
                    "severity": df["severity"],
                    "justification": df["justification"]
                })
            print(f"  Merged: {len(confirmed)} total findings going to integration")
        
        # ── Step 5: Cross-file integration pass ──
        print("\n--- Pass 3: Cross-File Integration ---")
        report = self.reviewer.integration_pass(confirmed)
        self.metrics["actionable_count"] = len(report.get("review_comments", []))
        self.metrics["passes_executed"].append("integration")
        
        # ── Final summary ──
        print(f"\n{'=' * 60}")
        print("PIPELINE COMPLETE")
        print(f"  Mode: {self.metrics['processing_mode']}")
        print(f"  Passes: {' -> '.join(self.metrics['passes_executed'])}")
        print(f"  Funnel: {self.metrics['detection_count']} detected"
              f" -> {self.metrics['confirmed_count']} confirmed"
              f" (+ {self.metrics['domain_findings']} domain)"
              f" -> {self.metrics['actionable_count']} actionable")
        
        if report.get("review_comments"):
            print(f"\nFinal Review Comments:")
            for i, c in enumerate(report["review_comments"], 1):
                print(f"  {i}. [{c['severity']}] {c['comment']}")
                print(f"     Files: {', '.join(c['files'])}")
        
        print(f"\nSummary: {report.get('summary', 'N/A')}")
        
        return {
            "report": report,
            "metrics": self.metrics.copy(),
            "processing_mode": self.metrics["processing_mode"]
        }


# ── Test: General review (no domain) ──
print("TEST 1: General Review")
print()
general_pipeline = ProductionPipeline(client, domain=None)
general_result = general_pipeline.process_pr(test_files)

print("\n\n")

# ── Test: Financial domain review ──
print("TEST 2: Financial Domain Review")
print()
financial_files = [
    {
        "filename": "payment_service.py",
        "content": '''def transfer_funds(from_acct, to_acct, amount):
    balance = from_acct.balance - amount
    if balance >= 0:
        from_acct.balance = balance
        to_acct.balance += amount

def calculate_tax(price, rate):
    return price * rate

def split_payment(total, ways):
    return total / ways'''
    },
    {
        "filename": "billing.py",
        "content": '''def generate_invoice(items):
    total = 0.0
    for item in items:
        total += item["price"] * item["quantity"]
    return {"total": total, "items": items}'''
    }
]

financial_pipeline = ProductionPipeline(client, domain="financial")
financial_result = financial_pipeline.process_pr(financial_files)

In [ ]:
#@title 🎧 Listen: Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_17_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Visualization: Multi-Pass Funnel & Cost Comparison

Let's visualize two key concepts:
1. **Left**: The multi-pass review funnel — how findings narrow at each stage
2. **Right**: Cost comparison between batch and real-time at different document scales

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ── Left: Multi-Pass Review Funnel ──
stages = ["Detection\n(High Recall)", "Severity\nAssessment", "Integration\n(Actionable)"]
counts = [6, 4, 2]
colors = ["#e74c3c", "#f39c12", "#27ae60"]

# Draw funnel as horizontal bars centered
bar_widths = counts
y_positions = [2, 1, 0]

for i, (stage, count, color) in enumerate(zip(stages, counts, colors)):
    bar = ax1.barh(y_positions[i], count, height=0.6, color=color, alpha=0.85,
                   edgecolor='white', linewidth=2)
    ax1.text(count / 2, y_positions[i], f"{count} findings",
             ha='center', va='center', fontsize=13, fontweight='bold', color='white')

ax1.set_yticks(y_positions)
ax1.set_yticklabels(stages, fontsize=11)
ax1.set_xlabel("Number of Findings", fontsize=12)
ax1.set_title("Multi-Pass Review Funnel", fontsize=14, fontweight='bold', pad=15)
ax1.set_xlim(0, 8)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Add annotations
ax1.annotate("False positives\nfiltered", xy=(5, 1.5), fontsize=9,
             fontstyle='italic', color='#666', ha='center')
ax1.annotate("Grouped &\ndeduplicated", xy=(4.5, 0.5), fontsize=9,
             fontstyle='italic', color='#666', ha='center')

# ── Right: Cost Comparison ──
doc_counts = [10, 50, 100, 500, 1000, 5000, 10000]
cost_per_doc = 0.02

realtime_costs = [n * cost_per_doc for n in doc_counts]
batch_costs = [n * cost_per_doc * 0.5 for n in doc_counts]
savings = [r - b for r, b in zip(realtime_costs, batch_costs)]

x = np.arange(len(doc_counts))
width = 0.35

bars1 = ax2.bar(x - width/2, realtime_costs, width, label='Real-Time', color='#e74c3c', alpha=0.85)
bars2 = ax2.bar(x + width/2, batch_costs, width, label='Batch (50% off)', color='#27ae60', alpha=0.85)

ax2.set_xlabel("Document Count", fontsize=12)
ax2.set_ylabel("Cost (USD)", fontsize=12)
ax2.set_title("Batch vs Real-Time Cost", fontsize=14, fontweight='bold', pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels([f"{n:,}" for n in doc_counts], rotation=45, ha='right')
ax2.legend(fontsize=11)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Add savings annotation for the largest bar
ax2.annotate(f"Save\n${savings[-1]:,.0f}",
             xy=(x[-1], batch_costs[-1]),
             xytext=(x[-1] - 0.8, realtime_costs[-1] * 0.85),
             fontsize=10, fontweight='bold', color='#27ae60',
             arrowprops=dict(arrowstyle='->', color='#27ae60', lw=1.5))

plt.tight_layout()
plt.savefig("batch_multipass_visualization.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nKey insight: At 10,000 documents, batch saves $100 compared to real-time.")
print("Multi-pass review reduces 6 raw findings to 2 actionable comments (67% noise reduction).")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_18_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Key Takeaways

### Batch API
1. **50% cost savings for non-urgent workloads.** If you can wait up to 24 hours, batch processing cuts your API costs in half.
2. **`custom_id` is your tracking mechanism.** Every batch request gets a custom_id that appears in the results, letting you match extractions back to source documents.
3. **Batch for overnight, real-time for interactive.** Nightly document processing, bulk extraction, and periodic reports belong in batch. User-facing chat, live code review, and interactive tools need real-time.
4. **Structure batch requests as JSONL.** Each line is a complete API request with custom_id and params.

### Multi-Pass Review
5. **Separate detection from severity assessment.** Detection casts a wide net (high recall). Severity filtering narrows to actionable findings (high precision).
6. **Cross-file integration catches what single-file review misses.** Related findings across files, null-check guarantees from callers, and systemic patterns all require cross-file context.
7. **Domain-specific passes add specialized expertise.** Financial code needs precision checks. Medical code needs HIPAA compliance. Security code needs vulnerability pattern matching.
8. **Multi-pass trades latency for quality.** Three passes take ~3x longer than one pass. Acceptable for CI/CD (background), not for interactive editors (foreground).

### Production Architecture
9. **Combine batch + multi-pass for maximum efficiency.** Use batch for the detection pass (highest volume, most parallelizable), real-time for severity and integration (need context from previous passes).
10. **Track metrics at every stage.** The funnel (detected -> confirmed -> actionable) tells you if your prompts are well-calibrated. Too many false positives? Tighten detection. Too few findings? Loosen it.

---

These techniques complete your prompt engineering toolkit. Combined with the criteria, few-shot examples, tool_use schemas, and validation-retry loops from Notebooks 01-03, you now have every pattern tested on the Claude Certified Architect exam.

---

*Built with Vizuara — AI education that meets you where you are.*